In [ ]:
# S1 -- paths, inputs and test settings (TEST split). Peak RAM: ~60 MB
from pathlib import Path
import time, hashlib
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon

project_dir = Path(r"D:\Project\Radiology_RAG")
TEST_DIR = project_dir/"results"/"test"
TAGS = {"qwen2.5:1.5b": "qwen25_1.5b", "llama3.2:3b": "llama32_3b"}
SCORED_FILES = {m: TEST_DIR/f"test_kb_v2_{t}_scored.csv" for m, t in TAGS.items()}       # from test_run.ipynb (S8)
REF_FILES    = {m: TEST_DIR/f"test_kb_v2_{t}_refmetrics.csv" for m, t in TAGS.items()}   # from reference_metrics.ipynb (R4)

BASELINE     = "no_rag"
RAG_METHODS  = ["oracle", "hybrid", "hybrid_anchored", "1hop", "2hop"]
MAIN_METHODS = ["hybrid", "1hop", "2hop"]          # the three methods in the research question
TEST_METRICS = ["summac_src_def", "summac_src_def_worst", "summac_ref",
                "term_kept", "term_glossed", "term_bare", "fkgl", "fre", "gold_cui_recall",
                "sari", "bleu", "rougeL", "bertscore_f1"]
MIN_SAMPLES = 5
md5 = lambda t: hashlib.md5(str(t).encode("utf-8")).hexdigest()

parts = []
for model, f in SCORED_FILES.items():
    if not f.exists():
        print(f"skipped {model}: {f.name} not found yet")
        continue
    df = pd.read_csv(f)
    df = df[df["model"] == model].reset_index(drop=True)
    assert df["sample_id"].nunique() == 2790 and len(df) == 2790 * 6, f"{model}: {len(df):,} rows, expected 16,740"
    assert df["summac_src_def"].notna().all(), f"{model}: SummaC missing for some answers"
    rf = REF_FILES[model]
    if rf.exists():
        ref = pd.read_csv(rf)
        df = df.merge(ref, on=["model", "method", "sample_id"], how="left", validate="one_to_one")
        same = df["answer"].fillna("").astype(str).map(md5) == df["answer_md5"]
        assert same.all(), f"{model}: reference metrics belong to different answers; re-run reference_metrics.ipynb"
    else:
        print(f"{model}: no reference metrics yet ({rf.name}); testing the other metrics only")
    parts.append(df)
assert parts, "no scored file found"
all_scored = pd.concat(parts, ignore_index=True)
SCORED_RUNS = {m: f for m, f in SCORED_FILES.items() if m in set(all_scored["model"])}   # S2 loops over these
print(all_scored.groupby("model")["sample_id"].nunique().rename("captions").to_string())

In [ ]:
# S2 -- Wilcoxon vs no_rag + Friedman / pairwise for the main methods, all Holm-corrected. ~5 s. Peak RAM: ~50 MB
def holm(pvals):
    """Holm-Bonferroni step-down adjustment."""
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    m, adj, running = len(pvals), np.empty(len(pvals)), 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * pvals[idx])
        adj[idx] = min(1.0, running)
    return adj

def paired(wide, a, b):
    d = (wide[a] - wide[b]).dropna()
    if len(d) < MIN_SAMPLES:
        return None
    p = 1.0 if np.allclose(d, 0) else wilcoxon(d).pvalue
    return {"n": len(d), "median_diff": d.median(), "mean_diff": d.mean(),
            "share_a_better": (d > 0).mean(), "p": p}

vs_base, omni_rows, pair_rows = [], [], []
for model in SCORED_RUNS:
    for metric in [m for m in TEST_METRICS if m in all_scored.columns]:
        wide = all_scored[all_scored["model"] == model].pivot_table(index="sample_id", columns="method", values=metric)

        # (1) each retrieval method vs no_rag (gold_cui_recall has no no_rag value, so it is skipped here)
        fam = []
        if BASELINE in wide and wide[BASELINE].notna().any():
            for m in RAG_METHODS:
                r = paired(wide, m, BASELINE)
                if r:
                    fam.append({"model": model, "metric": metric, "method": m} | r)
        if fam:
            fam = pd.DataFrame(fam); fam["p_holm"] = holm(fam["p"]); vs_base.append(fam)

        # (2) hybrid vs 1hop vs 2hop: Friedman, then pairwise Wilcoxon with Holm
        w3 = wide.reindex(columns=MAIN_METHODS).dropna()
        if len(w3) >= MIN_SAMPLES:
            chi2, p = friedmanchisquare(*[w3[m] for m in MAIN_METHODS])
            omni_rows.append({"model": model, "metric": metric, "n": len(w3), "friedman_chi2": chi2,
                              "p": p, "kendall_w": chi2 / (len(w3) * (len(MAIN_METHODS) - 1))})
            fam2 = [{"model": model, "metric": metric, "a": a, "b": b} | paired(w3, a, b)
                    for a, b in combinations(MAIN_METHODS, 2)]
            fam2 = pd.DataFrame(fam2); fam2["p_holm"] = holm(fam2["p"]); pair_rows.append(fam2)

vs_base_table  = pd.concat(vs_base, ignore_index=True).round(4)
friedman_table = pd.DataFrame(omni_rows).round(4)
pairwise_table = pd.concat(pair_rows, ignore_index=True).round(4)

print("(1) each method vs no_rag: Holm-corrected p  (see vs_base_table for mean_diff; > 0 = method higher)")
display(vs_base_table.pivot_table(index=["model", "method"], columns="metric", values="p_holm").round(3))
print("\n(2) Friedman: hybrid vs 1hop vs 2hop")
display(friedman_table)
display(pairwise_table)

test_dir = project_dir/"results"/"runs"/(time.strftime("%Y%m%d_%H%M%S") + "_tests")
test_dir.mkdir(parents=True, exist_ok=True)
vs_base_table.to_csv(test_dir/"vs_no_rag_wilcoxon_holm.csv", index=False)
friedman_table.to_csv(test_dir/"friedman.csv", index=False)
pairwise_table.to_csv(test_dir/"pairwise_wilcoxon_holm.csv", index=False)
(test_dir/"inputs.txt").write_text("\n".join(f"{m}: {f}" for m, f in SCORED_RUNS.items()), encoding="utf-8")
print("saved to", test_dir)